# 07 — Explainable AI (SHAP + Grad-CAM)
**RoadSentinel AI** — High-trust diagnostic interpretability.
Uses SHAP (TreeExplainer) on tabular triage classifiers and Grad-CAM on vision keyframes.

In [ ]:
import os, sys, cv2
sys.path.insert(0, '..')
import pandas as pd, joblib, matplotlib.pyplot as plt
from src.xai import explain_tabular_model, explain_frame_gradcam
from src.preprocessing import split_first

## 1. Tabular Model Explainability: SHAP Summary Plot

In [ ]:
model_path = '../models/xgb_triage_pipeline.joblib'
if not os.path.exists(model_path):
    from scripts.train_models import run_training_pipeline
    run_training_pipeline()

xgb_pipe = joblib.load(model_path)
df = pd.read_csv('../data/engineered_features.csv')
_, X_test, _, y_test = split_first(df)
sample_X = X_test.sample(150, random_state=42)

shap_path = explain_tabular_model(xgb_pipe, sample_X, save_path='../models/metrics/shap_summary.png')
print(f"SHAP summary saved -> {shap_path}")

## 2. Computer Vision Explainability: Grad-CAM Attention Heatmaps

In [ ]:
sample_frame_path = '../demo/clip_1_frame.jpg'
if os.path.exists(sample_frame_path):
    frame_bgr = cv2.imread(sample_frame_path)
    overlay = explain_frame_gradcam(frame_bgr, save_path='../demo/precomputed/gradcam/clip_1_gradcam.png')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    ax1.set_title('Original Incident Keyframe')
    ax1.axis('off')
    
    ax2.imshow(overlay)
    ax2.set_title('Grad-CAM Attention Heatmap Overlay')
    ax2.axis('off')
    plt.show()